Pretrained models

<style>
    .heatMap {
        width: 70%;
        text-align: center;
    }
    .heatMap th {
        color: #ffffff;
        background: #444444;
        word-wrap: break-word;
        text-align: center;
    }
    .heatMap tr:nth-child(odd) {
        color: #ffffff;
        background: #666666;
    }
    .heatMap tr:nth-child(even) { 
        color: #ffffff;
        background: #444444; }
</style>

<div class="heatMap">

| Model Name | Performance<br>Sentence Embedding<br>(14 Datasets) | Performance<br>Semantic Search<br>(6 Datasets) | Avg. Performance | Speed | Model Size |
| ---------: | :------------------------------------------------: | :--------------------------------------------: | :--------------: | :---: | :--------: |
| all-mpnet-base-v2 | 69.57 | 57.02 | 63.30 | 2800 | 420 MB | 
| multi-qa-mpnet-base-dot-v1 | 	66.76 |	57.60 |	62.18 |	2800 |	420 MB | 
| all-distilroberta-v1 | 	68.73 |	50.94 |	59.84 |	4000 |	290 MB |
| all-MiniLM-L12-v2 | 	68.70 |	50.82 |	59.76 |	7500 |	120 MB |
| multi-qa-distilbert-cos-v1 | 	65.98 |	52.83 |	59.41 |	4000 |	250 MB |
| all-MiniLM-L6-v2 | 	68.06 |	49.54 |	58.80 |	14200 |	80 MB |
| multi-qa-MiniLM-L6-cos-v1 | 	64.33 |	51.83 |	58.08 |	14200 |	80 MB |
| paraphrase-multilingual-mpnet-base-v2 |  	65.83 |	41.68 |	53.75 |	2500 |	970 MB |
| paraphrase-albert-small-v2 | 	64.46 |	40.04 |	52.25 |	5000 |	43 MB |
| paraphrase-multilingual-MiniLM-L12-v2 |  	64.25 |	39.19 |	51.72 |	7500 |	420 MB |
| paraphrase-MiniLM-L3-v2 | 	62.29 |	39.19 |	50.74 |	19000 |	61 MB |
| distiluse-base-multilingual-cased-v1 | 	61.30 |	29.87 |	45.59 |	4000 |	480 MB |
| distiluse-base-multilingual-cased-v2 | 	60.18 |	27.35 |	43.77 |	4000 |	480 MB |
</div>

**Checked last time on 15/01/2024**.

For a complete list of the supported models check [here](https://www.sbert.net/docs/pretrained_models.html).

In [1]:
import pandas as pd

from nb_config import RAW_DATA_PATH

df = pd.read_parquet(RAW_DATA_PATH + 'movie_descriptors.parquet')

In [2]:
sentences = df.descriptor.tolist()

In [3]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')


#Compute embeddings
embeddings = model.encode(sentences, convert_to_tensor=True, show_progress_bar=True)

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

In [50]:
query = "An innocent man in prison that never loses the hope starts helping the warden as accountant"

query_embedding = model.encode(query, convert_to_tensor=True)


#Compute cosine-similarities for each sentence with each other sentence
cosine_scores = util.cos_sim(query_embedding, embeddings)

In [51]:
scores = cosine_scores.tolist()[0]
df.loc[:, 'scores'] = scores
df.sort_values(by='scores', ascending=False).head(10)

,title,release_year,descriptor,scores
71,The Shawshank Redemption,1994,Framed in the 1940s for the double murder of h...,0.424905
2756,Central Intelligence,2016,After he reunites with an old pal through Face...,0.419534
1678,Bug,2006,"A lonely waitress with a tragic past, Agnes ro...",0.417279
760,Midnight Run,1988,An accountant embezzles $15 million of mob mon...,0.413500
2303,A Coffee in Berlin,2012,A fateful day pushes an aimless college dropou...,0.407613
2716,Eddie the Eagle,2016,"Inspired by true events, Eddie the Eagle is a ...",0.403517
462,Midnight Cowboy,1969,A naive male prostitute and his sickly friend ...,0.397446
777,Papillon,1973,A man befriends a fellow criminal as the two o...,0.392417
1511,Stay,2005,"Psychiatrist Sam Foster has a new patient, Hen...",0.388520
1451,Hostage,2005,When a mafia accountant is taken hostage on hi...,0.387474


In [6]:
df.loc[df.title == "The Shawshank Redemption", 'descriptor'].item()

'Framed in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope.'

In [7]:
import torch

print("Torch version:",torch.__version__)

print("Is CUDA enabled?",torch.cuda.is_available())

Torch version: 2.1.2
Is CUDA enabled? True


In [8]:
cuda = torch.device

In [20]:


from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('msmarco-distilbert-cos-v5', device='cuda')


#Compute embeddings
embeddings = model.encode(sentences, convert_to_tensor=True, show_progress_bar=True)

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

In [27]:
query = "Movie about an innocent man in prison that starts helping the warden as accountant and never loses the hope"

query_embedding = model.encode(query, convert_to_tensor=True)


#Compute cosine-similarities for each sentence with each other sentence
cosine_scores = util.cos_sim(query_embedding, embeddings)

In [28]:
scores = cosine_scores.tolist()[0]
df.loc[:, 'scores'] = scores
df.sort_values(by='scores', ascending=False).head(15)

,title,release_year,descriptor,scores
2303,A Coffee in Berlin,2012,A fateful day pushes an aimless college dropou...,0.439840
71,The Shawshank Redemption,1994,Framed in the 1940s for the double murder of h...,0.404926
462,Midnight Cowboy,1969,A naive male prostitute and his sickly friend ...,0.386168
2716,Eddie the Eagle,2016,"Inspired by true events, Eddie the Eagle is a ...",0.383909
198,To Catch a Thief,1955,A delightful Hitchcock film about an ex-burgla...,0.380561
44,Strange Days,1995,Set in the year 1999 during the last days of t...,0.375328
777,Papillon,1973,A man befriends a fellow criminal as the two o...,0.372944
1451,Hostage,2005,When a mafia accountant is taken hostage on hi...,0.372716
477,Halloween,1978,"In John Carpenter's horror classic, a psychoti...",0.372711
102,The Firm,1993,Mitch McDeere is a young man with a promising ...,0.370128
